In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import re
import sys

sys.path.insert(0, str(Path.cwd() / 'src'))
from study2_utils import (
    CONTROL_GROUP, EXPERIMENTAL_GROUP, EXCLUDED_PARTICIPANTS,
    assign_group, ttest, analyze_bart, analyze_emotion_gonogo,
)


In [ ]:
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR.parent.parent / 'data' / 'study2' / 'behavior'
RESULTS_DIR = NOTEBOOK_DIR / 'analysis_results'
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
# preprocessing 

preform = pd.read_excel(DATA_DIR / 'googleform' / 'BRAIN-PT_preform.xlsx')
postform = pd.read_excel(DATA_DIR / 'googleform' / 'BRAIN-PT_postform.xlsx')
takehome = pd.read_excel(DATA_DIR / 'googleform' / 'BRAIN-PT takehome.xlsx')

def extract_participant_num_preform(name_str):
    if pd.isna(name_str):
        return None
    match = re.match(r'(\d+)_', str(name_str))
    if match:
        return int(match.group(1))
    return None

def extract_participant_num_postform(name_str):
    if pd.isna(name_str):
        return None
    match = re.match(r'(\d+)_', str(name_str))
    if match:
        return int(match.group(1)) - 100  
    return None

def extract_name_only(name_str):
    if pd.isna(name_str):
        return None
    match = re.match(r'\d+_(.+)', str(name_str))
    if match:
        return match.group(1)
    if ',' in str(name_str):
        return str(name_str).split(',')[0].strip()
    return str(name_str)

def extract_adhd_score(response_str):
    if pd.isna(response_str):
        return None
    match = re.match(r'(\d+)', str(response_str))
    if match:
        return int(match.group(1))
    return None

name_col_pre = '성함을 입력해주세요. (ex. 홍길동) '
preform['participant_num'] = preform[name_col_pre].apply(extract_participant_num_preform)
preform['name'] = preform[name_col_pre].apply(extract_name_only)
preform['group'] = preform['participant_num'].apply(assign_group)
preform = preform[~preform['participant_num'].isin(EXCLUDED_PARTICIPANTS)].reset_index(drop=True)

name_col_post = '당신의 성함을 입력해주세요. '
postform['participant_num'] = postform[name_col_post].apply(extract_participant_num_postform)
postform['name'] = postform[name_col_post].apply(extract_name_only)
postform['group'] = postform['participant_num'].apply(assign_group)
postform = postform[~postform['participant_num'].isin(EXCLUDED_PARTICIPANTS)].reset_index(drop=True)

name_col_takehome = '성함과 전화번호 뒷자리를 작성해주세요. (ex. 홍길동, 1234)'
takehome['name'] = takehome[name_col_takehome].apply(extract_name_only)

name_to_num = dict(zip(preform['name'], preform['participant_num']))
takehome['participant_num'] = takehome['name'].map(name_to_num)
takehome['group'] = takehome['participant_num'].apply(assign_group)
takehome = takehome[~takehome['participant_num'].isin(EXCLUDED_PARTICIPANTS)].reset_index(drop=True)

print('Preform n:', preform['participant_num'].nunique(), 'Postform n:', postform['participant_num'].nunique(), 'Takehome n:', takehome['participant_num'].nunique())

instagram_col = '스크린타임에서 인스타그램 사용 시간을 캡쳐해서 업로드해주세요. '
youtube_col = '스크린타임에서 유튜브 사용 시간을 캡쳐해서 업로드해주세요. '
instagram_pct_col = '당신의 총 인스타그램 이용 시간에 비해, 릴스 시청 시간은 몇 퍼센트 정도 차지한다고 생각하십니까? '
youtube_pct_col = '당신의 유튜브 총 이용 시간에 비해, 쇼츠 시청 시간은 몇 퍼센트 정도 차지한다고 생각하십니까? '

preform['instagram_minutes'] = preform[instagram_col]
preform['youtube_minutes'] = preform[youtube_col]
preform['instagram_reels_ratio'] = preform[instagram_pct_col]  # already a 0-1 ratio
preform['youtube_shorts_ratio'] = preform[youtube_pct_col]      # already a 0-1 ratio
preform['reels_minutes'] = (preform['instagram_minutes'] * preform['instagram_reels_ratio']).round().astype('Int64')
preform['shorts_minutes'] = (preform['youtube_minutes'] * preform['youtube_shorts_ratio']).round().astype('Int64')
preform['total_shortform_minutes'] = preform['reels_minutes'] + preform['shorts_minutes']

postform['instagram_minutes'] = postform[instagram_col]
postform['youtube_minutes'] = postform[youtube_col]
postform['instagram_reels_ratio'] = postform[instagram_pct_col]  # already a 0-1 ratio
postform['youtube_shorts_ratio'] = postform[youtube_pct_col]      # already a 0-1 ratio
postform['reels_minutes'] = (postform['instagram_minutes'] * postform['instagram_reels_ratio']).round().astype('Int64')
postform['shorts_minutes'] = (postform['youtube_minutes'] * postform['youtube_shorts_ratio']).round().astype('Int64')
postform['total_shortform_minutes'] = postform['reels_minutes'] + postform['shorts_minutes']

adhd_question_cols = [col for col in preform.columns if re.match(r'\d+\.', col)]
preform['adhd_score'] = preform[adhd_question_cols].map(extract_adhd_score).sum(axis=1)
postform['adhd_score'] = postform[adhd_question_cols].map(extract_adhd_score).sum(axis=1)

print('Pre shortform (min):', round(preform['total_shortform_minutes'].mean(), 1))
print('Post shortform (min):', round(postform['total_shortform_minutes'].mean(), 1))
print('Pre ADHD:', round(preform['adhd_score'].mean(), 2))
print('Post ADHD:', round(postform['adhd_score'].mean(), 2))

In [ ]:
cols = ['participant_num', 'group', 'adhd_score', 'reels_minutes', 'shorts_minutes', 'total_shortform_minutes']

pre_summary = preform[preform['group'] != 'Unknown'][cols].copy()
pre_summary.insert(0, 'session', 'pre')

post_summary = postform[postform['group'] != 'Unknown'][cols].copy()
post_summary.insert(0, 'session', 'post')

summary_df = pd.concat([pre_summary, post_summary], ignore_index=True).sort_values(
    ['participant_num', 'session'], ascending=[True, True]
).reset_index(drop=True)

firebase = pd.read_excel(DATA_DIR / 'GameResults_cleaned.xlsx')

def extract_firebase_participant_num(name):
    if pd.isna(name):
        return None
    match = re.match(r'(\d+)_', str(name))
    if match:
        return int(match.group(1))
    return None

firebase['participant_num'] = firebase['participantName'].apply(extract_firebase_participant_num)
firebase['group'] = firebase['participant_num'].apply(assign_group)
firebase = firebase[~firebase['participant_num'].isin(EXCLUDED_PARTICIPANTS)].reset_index(drop=True)

print('Participants:', firebase['participant_num'].nunique())
print('Records:', len(firebase))

In [ ]:
exp1_files = sorted((DATA_DIR / 'brainpt_psyexp1').glob('*.csv'))
exp2_files = sorted((DATA_DIR / 'brainpt_psyexp2').glob('*.csv'))

def extract_participant_info_psychopy(filename):
    match = re.match(r'(\d+)_([a-zA-Z]+)_', filename.stem)
    if match:
        num_raw = int(match.group(1))
        name = match.group(2).lower()
        if num_raw >= 100:
            num = num_raw - 100
        else:
            num = num_raw
        return num, name
    return None, None

exp1_by_name = {}
for f in exp1_files:
    num, name = extract_participant_info_psychopy(f)
    if name:
        exp1_by_name[name] = {'file': f, 'num': num}

exp2_by_name = {}
for f in exp2_files:
    num, name = extract_participant_info_psychopy(f)
    if name:
        exp2_by_name[name] = {'file': f, 'num': num}

matched_participants = set(exp1_by_name.keys()) & set(exp2_by_name.keys())

bart_exp1_results = []
bart_exp2_results = []
gonogo_exp1_results = []
gonogo_exp2_results = []

for name in sorted(matched_participants):
    num = exp1_by_name[name]['num']
    if num in EXCLUDED_PARTICIPANTS:
        continue
    group = assign_group(num)
    
    # Exp1
    df1 = pd.read_csv(exp1_by_name[name]['file'])
    bart1 = analyze_bart(df1, num)
    if bart1:
        bart1.update({'name': name, 'session': 'exp1', 'group': group})
        bart_exp1_results.append(bart1)
    
    gonogo1 = analyze_emotion_gonogo(df1, num)
    if gonogo1:
        gonogo1.update({'name': name, 'session': 'exp1', 'group': group})
        gonogo_exp1_results.append(gonogo1)
    
    # Exp2
    df2 = pd.read_csv(exp2_by_name[name]['file'])
    bart2 = analyze_bart(df2, num)
    if bart2:
        bart2.update({'name': name, 'session': 'exp2', 'group': group})
        bart_exp2_results.append(bart2)
    
    gonogo2 = analyze_emotion_gonogo(df2, num)
    if gonogo2:
        gonogo2.update({'name': name, 'session': 'exp2', 'group': group})
        gonogo_exp2_results.append(gonogo2)

bart_exp1_df = pd.DataFrame(bart_exp1_results)
bart_exp2_df = pd.DataFrame(bart_exp2_results)
gonogo_exp1_df = pd.DataFrame(gonogo_exp1_results)
gonogo_exp2_df = pd.DataFrame(gonogo_exp2_results)

bart_comparison = bart_exp1_df[['participant_num', 'group', 'total_earnings', 'mean_pumps', 'explosion_rate', 'mean_pump_latency']].merge(
    bart_exp2_df[['participant_num', 'total_earnings', 'mean_pumps', 'explosion_rate', 'mean_pump_latency']],
    on='participant_num',
    suffixes=('_exp1', '_exp2')
)
bart_comparison['earnings_change'] = bart_comparison['total_earnings_exp2'] - bart_comparison['total_earnings_exp1']
bart_comparison['mean_pumps_change'] = bart_comparison['mean_pumps_exp2'] - bart_comparison['mean_pumps_exp1']
bart_comparison['pump_latency_change'] = bart_comparison['mean_pump_latency_exp2'] - bart_comparison['mean_pump_latency_exp1']
bart_comparison['explosion_rate_change'] = bart_comparison['explosion_rate_exp2'] - bart_comparison['explosion_rate_exp1']

gonogo_comparison = gonogo_exp1_df[['participant_num', 'group', 'accuracy', 'go_accuracy', 'nogo_accuracy', 'mean_rt']].merge(
    gonogo_exp2_df[['participant_num', 'accuracy', 'go_accuracy', 'nogo_accuracy', 'mean_rt']],
    on='participant_num',
    suffixes=('_exp1', '_exp2')
)
gonogo_comparison['accuracy_change'] = gonogo_comparison['accuracy_exp2'] - gonogo_comparison['accuracy_exp1']
gonogo_comparison['rt_change'] = gonogo_comparison['mean_rt_exp2'] - gonogo_comparison['mean_rt_exp1']

In [ ]:
control_pre = preform[preform['group'] == 'Control']
exp_pre = preform[preform['group'] == 'Experimental']

ttest(control_pre['total_shortform_minutes'], exp_pre['total_shortform_minutes'], ('con','exp'))
pd.concat([control_pre, exp_pre])[['participant_num','group','total_shortform_minutes']].to_excel(RESULTS_DIR / 'q1_1_shortform_pre.xlsx', index=False)

In [ ]:
ttest(control_pre['adhd_score'], exp_pre['adhd_score'], ('con','exp'))
pd.concat([control_pre, exp_pre])[['participant_num','group','adhd_score']].to_excel(RESULTS_DIR / 'q1_2_adhd_pre.xlsx', index=False)

Q1-3. Google Form: Did the experimental group's shortform usage time decrease significantly more than the control group's from Pre to Post?

- 2-way mixed ANOVA
- Independent t-test on change scores (comparing the amount of change between groups)


In [ ]:
matched_data = preform[['participant_num','total_shortform_minutes','group']].merge(
    postform[['participant_num','total_shortform_minutes']], on='participant_num', suffixes=('_pre','_post'))
matched_data['change'] = matched_data['total_shortform_minutes_post'] - matched_data['total_shortform_minutes_pre']

c_chg = matched_data[matched_data['group']=='Control']['change'].dropna()
e_chg = matched_data[matched_data['group']=='Experimental']['change'].dropna()
ttest(c_chg, e_chg, ('con','exp'))
matched_data[['participant_num','group','total_shortform_minutes_pre','total_shortform_minutes_post','change']].to_excel(RESULTS_DIR / 'q1_3_shortform_change.xlsx', index=False)

In [ ]:
control_gonogo = gonogo_comparison[gonogo_comparison['group'] == 'Control']
exp_gonogo   = gonogo_comparison[gonogo_comparison['group'] == 'Experimental']

ttest(control_gonogo['accuracy_change'], exp_gonogo['accuracy_change'], ('con','exp'))
gonogo_comparison[['participant_num','group','accuracy_change']].to_excel(RESULTS_DIR / 'q1_4_accuracy_change.xlsx', index=False)

In [ ]:
ttest(control_gonogo['rt_change'], exp_gonogo['rt_change'], ('con','exp'))
gonogo_comparison[['participant_num','group','rt_change']].to_excel(RESULTS_DIR / 'q1_5_rt_change.xlsx', index=False)

In [ ]:
control_bart = bart_comparison[bart_comparison['group'] == 'Control']
exp_bart = bart_comparison[bart_comparison['group'] == 'Experimental']

ttest(control_bart['pump_latency_change'], exp_bart['pump_latency_change'], ('con','exp'))
bart_comparison[['participant_num','group','mean_pump_latency_exp1','mean_pump_latency_exp2','pump_latency_change']].to_excel(RESULTS_DIR / 'q1_6_pump_latency_change.xlsx', index=False)

In [ ]:
ttest(control_bart['mean_pumps_change'], exp_bart['mean_pumps_change'], ('con','exp'))
bart_comparison[['participant_num','group','mean_pumps_exp1','mean_pumps_exp2','mean_pumps_change']].to_excel(RESULTS_DIR / 'q1_7_mean_pumps_change.xlsx', index=False)

In [ ]:
ttest(control_bart['explosion_rate_change'], exp_bart['explosion_rate_change'], ('con','exp'))
bart_comparison[['participant_num','group','explosion_rate_exp1','explosion_rate_exp2','explosion_rate_change']].to_excel(RESULTS_DIR / 'q1_8_explosion_rate_change.xlsx', index=False)

In [ ]:
exp_matched = matched_data[matched_data['group'] == 'Experimental']
pre, post = exp_matched['total_shortform_minutes_pre'], exp_matched['total_shortform_minutes_post']
ttest(pre, post, ('pre','post'), paired=True)
exp_matched[['participant_num','group','total_shortform_minutes_pre','total_shortform_minutes_post']].to_excel(RESULTS_DIR / 'q2_1_exp_shortform.xlsx', index=False)

In [ ]:
adhd_matched = preform[['participant_num','adhd_score','group']].merge(
    postform[['participant_num','adhd_score']], on='participant_num', suffixes=('_pre','_post'))
exp_adhd = adhd_matched[adhd_matched['group'] == 'Experimental']
ttest(exp_adhd['adhd_score_pre'], exp_adhd['adhd_score_post'], ('pre','post'), paired=True)
exp_adhd[['participant_num','group','adhd_score_pre','adhd_score_post']].to_excel(RESULTS_DIR / 'q2_2_exp_adhd.xlsx', index=False)

Q2-3. Google Form (Take-home): Did scores on the 5 questions change significantly?

(1) Day 1 vs Day 14 comparison

Paired t-test for each question


In [ ]:
question_cols = [
    '오늘 하루, 집중해야 하는 일을 끝까지 할 수 있었다. ',
    '오늘 하루, 기분이 크게 흔들리지 않고 안정적이었다. ',
    '오늘 하루, 하고 싶은 걸 바로 하지 않고 한 번 더 생각할 수 있었다. ',
    '오늘 하루, 내가 세운 계획을 대부분 지켰다. ',
    '오늘 하루, 전반적으로 만족스러운 하루를 보냈다. '
]
takehome_exp = takehome[takehome['group'] == 'Experimental']
day1  = takehome_exp[takehome_exp['day'] == 1]
day14 = takehome_exp[takehome_exp['day'] == 14]
common = sorted(set(day1['participant_num']) & set(day14['participant_num']))

for i, col in enumerate(question_cols, 1):
    v1  = [day1[day1['participant_num']==p][col].values[0]  for p in common]
    v14 = [day14[day14['participant_num']==p][col].values[0] for p in common]
    t, p = stats.ttest_rel(v1, v14)
    print(f'Q{i}: {np.mean(v1):.2f}→{np.mean(v14):.2f}  t={t:.3f} p={p:.4f} {"*" if p<0.05 else "ns"}')
    pd.DataFrame({'participant_num': common, 'day1': v1, 'day14': v14}).to_excel(RESULTS_DIR / f'q2_3_takehome_Q{i}.xlsx', index=False)

(2) Day 1-14 trend analysis

**Analysis:** Linear regression (for each question)


In [ ]:
from scipy.stats import linregress

for i, col in enumerate(question_cols, 1):
    daily = takehome_exp.groupby('day')[col].mean()
    s, intercept, r, p, _ = linregress(daily.index.astype(float), daily.values.astype(float))
    print(f'Q{i}: slope={s:.3f}  R²={r**2:.3f}  p={p:.4f} {"*" if p<0.05 else "ns"}')

In [ ]:
from scipy.stats import linregress

q_labels = ["Q1. Focus", "Q2. Mood stability", "Q3. Impulse control", "Q4. Plan adherence", "Q5. Overall satisfaction"]
groups = [("Experimental", "Experimental"), ("Control", "Control")]

print("=== Day 1 vs Day 14 Paired t-test ===")
for group_key, group_label in groups:
    df_g = takehome[takehome["group"] == group_key]
    d1  = df_g[df_g["day"] == 1]
    d14 = df_g[df_g["day"] == 14]
    common = sorted(set(d1["participant_num"]) & set(d14["participant_num"]))
    print(f"[{group_label}] n={len(common)}")
    for qi, (col, ql) in enumerate(zip(question_cols, q_labels), 1):
        v1  = np.array(pd.to_numeric([d1[d1["participant_num"]==p][col].values[0]  for p in common], errors="coerce"), dtype=float)
        v14 = np.array(pd.to_numeric([d14[d14["participant_num"]==p][col].values[0] for p in common], errors="coerce"), dtype=float)
        t, p = stats.ttest_rel(v1, v14)
        sig = "*" if p < 0.05 else "ns"
        print(f"  {ql}: {np.nanmean(v1):.2f} -> {np.nanmean(v14):.2f}  t={t:.3f} p={p:.4f} {sig}")

print("=== Linear Trend (Day 1~14) ===")
for group_key, group_label in groups:
    df_g = takehome[takehome["group"] == group_key]
    print(f"[{group_label}]")
    for qi, (col, ql) in enumerate(zip(question_cols, q_labels), 1):
        daily = df_g.groupby("day")[col].apply(lambda x: pd.to_numeric(x, errors="coerce").mean())
        s, intercept, r, p, _ = linregress(daily.index.astype(float), daily.values.astype(float))
        sig = "*" if p < 0.05 else "ns"
        print(f"  {ql}: slope={s:.4f}  R2={r**2:.3f}  p={p:.4f} {sig}")


In [ ]:
from scipy import stats as _st

q_labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]
groups = [("Experimental", "Exp"), ("Control", "Con")]

fig, axes = plt.subplots(5, 2, figsize=(14, 22), sharey=True, sharex=True)
fig.suptitle("Takehome Day 1~14", fontsize=14, y=1.01)

for col_idx, (group_key, group_label) in enumerate(groups):
    df_group = takehome[takehome["group"] == group_key].copy()
    participants = df_group["participant_num"].dropna().unique()

    for row_idx, (col, q_label) in enumerate(zip(question_cols, q_labels)):
        ax = axes[row_idx, col_idx]

        for p in participants:
            df_p = df_group[df_group["participant_num"] == p].sort_values("day")
            if len(df_p) >= 2:
                ax.plot(df_p["day"], pd.to_numeric(df_p[col], errors="coerce"),
                        alpha=0.35, linewidth=1.1, marker="o", markersize=2.5)

        # 95% CI band
        def ci95(x):
            x = pd.to_numeric(x, errors="coerce").dropna()
            if len(x) < 2:
                return pd.Series({"mean": x.mean(), "lo": x.mean(), "hi": x.mean()})
            n = len(x)
            se = x.std(ddof=1) / np.sqrt(n)
            t_crit = _st.t.ppf(0.975, df=n-1)
            return pd.Series({"mean": x.mean(), "lo": x.mean() - t_crit*se, "hi": x.mean() + t_crit*se})

        ci = df_group.groupby("day")[col].apply(ci95).unstack()
        ax.fill_between(ci.index, ci["lo"], ci["hi"], alpha=0.25, color="black", label="95% CI")
        ax.plot(ci.index, ci["mean"], color="black", linewidth=2.5, label="avg", zorder=5)

        ax.set_title(f"{group_label} — {q_label}", fontsize=10)
        ax.set_xticks(range(1, 15))
        ax.set_ylim(0.5, 5.5)
        ax.set_ylabel("score (1~5)" if col_idx == 0 else "")
        if row_idx == 4:
            ax.set_xlabel("Day")

plt.tight_layout()
plt.show()


Q2-4. Firebase: Did game play count (proxy for shortform usage time) decrease significantly?

(1) Day 1 vs Day 14 comparison

Paired t-test (experimental group only)


In [ ]:
firebase_exp = firebase[firebase['group'] == 'Experimental'].copy()
day_col = 'iteration' if 'day' not in firebase_exp.columns else 'day'
firebase_exp['day'] = firebase_exp[day_col]

day1_counts  = firebase_exp[firebase_exp['day']==1].groupby('participant_num').size()
day14_counts = firebase_exp[firebase_exp['day']==14].groupby('participant_num').size()
common = sorted(set(day1_counts.index) & set(day14_counts.index))

if common:
    v1  = [day1_counts[p]  for p in common]
    v14 = [day14_counts[p] for p in common]
    t, p = stats.ttest_rel(v1, v14)
    print(f'Day1: {np.mean(v1):.2f}±{np.std(v1):.2f}  Day14: {np.mean(v14):.2f}±{np.std(v14):.2f}  t={t:.3f} p={p:.4f} {"*" if p<0.05 else "ns"}')
else:
    print('No common participants for day1/14')


(3) Day 1-14 trend analysis

**Analysis:** Linear regression (average across all participants)


In [ ]:
from scipy.stats import linregress

daily_avg = firebase_exp.groupby('day').size() / firebase_exp.groupby('day')['participant_num'].nunique()
s, intercept, r, p, _ = linregress(daily_avg.index.astype(float), daily_avg.values.astype(float))
print(f'slope={s:.3f}  R²={r**2:.3f}  p={p:.4f} {"*" if p<0.05 else "ns"}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(daily_avg.index, daily_avg.values, s=80)
ax.plot(daily_avg.index, s*daily_avg.index + intercept, 'r--')
ax.set_xlabel('Day'); ax.set_ylabel('mean play count')
ax.set_title(f'Q2-4 Firebase trend (slope={s:.3f})')
plt.tight_layout(); plt.show()

---
### Analysis 3: Individual differences (High/Low shortform usage time)
---


3-0. Split the experimental group into High/Low shortform viewing time groups

**Method:** Split by the median of Pre shortform usage time


In [ ]:
exp_preform = preform[preform['group'] == 'Experimental'].copy()
median_time = exp_preform['total_shortform_minutes'].median()
exp_preform['usage_group'] = exp_preform['total_shortform_minutes'].apply(lambda x: 'High' if x >= median_time else 'Low')

high_participants = exp_preform[exp_preform['usage_group']=='High']['participant_num'].tolist()
low_participants  = exp_preform[exp_preform['usage_group']=='Low']['participant_num'].tolist()
print(f'median: {median_time}min  High(n={len(high_participants)}): {sorted(high_participants)}  Low(n={len(low_participants)}): {sorted(low_participants)}')


Q3-1. Google Form: Did the High experimental group decrease shortform viewing time significantly more than the Low experimental group from Pre to Post?

**Analysis:** Independent t-test on change scores (High vs Low)


In [ ]:
matched_data_with_group = matched_data.copy()
matched_data_with_group['usage_group'] = matched_data_with_group['participant_num'].apply(
    lambda x: 'High' if x in high_participants else ('Low' if x in low_participants else None))

h_chg = matched_data_with_group[matched_data_with_group['usage_group']=='High']['change'].dropna()
l_chg = matched_data_with_group[matched_data_with_group['usage_group']=='Low']['change'].dropna()
ttest(h_chg, l_chg, ('High','Low'))
matched_data_with_group[matched_data_with_group['usage_group'].notna()][['participant_num','group','usage_group','change']].to_excel(RESULTS_DIR / 'q3_1_highlow_change.xlsx', index=False)

Q3-2. PsychoPy: Did the High experimental group's task accuracy increase significantly more than the Low experimental group's from Pre to Post?

**Analysis:** Independent t-test on accuracy change (High vs Low)


In [ ]:
gonogo_comparison_exp = gonogo_comparison[gonogo_comparison['group'] == 'Experimental'].copy()
gonogo_comparison_exp['usage_group'] = gonogo_comparison_exp['participant_num'].apply(
    lambda x: 'High' if x in high_participants else 'Low')

ttest(gonogo_comparison_exp[gonogo_comparison_exp['usage_group']=='High']['accuracy_change'],
      gonogo_comparison_exp[gonogo_comparison_exp['usage_group']=='Low']['accuracy_change'], ('High','Low'))
gonogo_comparison_exp[['participant_num','group','usage_group','accuracy_change']].to_excel(RESULTS_DIR / 'q3_2_accuracy_highlow.xlsx', index=False)

Q3-3. PsychoPy: Was task RT significantly faster in the High experimental group than the Low experimental group from Pre to Post?

**Analysis:** Independent t-test on RT change (High vs Low)


In [ ]:
ttest(gonogo_comparison_exp[gonogo_comparison_exp['usage_group']=='High']['rt_change'],
      gonogo_comparison_exp[gonogo_comparison_exp['usage_group']=='Low']['rt_change'], ('High','Low'))
gonogo_comparison_exp[['participant_num','group','usage_group','rt_change']].to_excel(RESULTS_DIR / 'q3_3_rt_highlow.xlsx', index=False)

Q3-4. Firebase: Did game play count decrease significantly more over time in the High experimental group than the Low experimental group?

Linear regression slope comparison (High vs Low)


In [ ]:
from scipy.stats import linregress

firebase_exp['usage_group'] = firebase_exp['participant_num'].apply(
    lambda x: 'High' if x in high_participants else ('Low' if x in low_participants else None))

for grp in ['High', 'Low']:
    sub = firebase_exp[firebase_exp['usage_group'] == grp]
    avg = sub.groupby('day').size() / sub.groupby('day')['participant_num'].nunique()
    s, intercept, r, p, _ = linregress(avg.index.astype(float), avg.values.astype(float))
    print(f'{grp}: slope={s:.3f}  R²={r**2:.3f}  p={p:.4f} {"*" if p<0.05 else "ns"}')

Q3-5. PsychoPy: Did pump latency change significantly more in the High experimental group than the Low experimental group from Pre to Post?

**Analysis:** Independent t-test on pump_latency_change (High vs Low)


In [ ]:
bart_comparison_exp = bart_comparison[bart_comparison['group'] == 'Experimental'].copy()
bart_comparison_exp['usage_group'] = bart_comparison_exp['participant_num'].apply(
    lambda x: 'High' if x in high_participants else 'Low')

ttest(bart_comparison_exp[bart_comparison_exp['usage_group']=='High']['pump_latency_change'],
      bart_comparison_exp[bart_comparison_exp['usage_group']=='Low']['pump_latency_change'], ('High','Low'))
bart_comparison_exp[['participant_num','group','usage_group','mean_pump_latency_exp1','mean_pump_latency_exp2','pump_latency_change']].to_excel(RESULTS_DIR / 'q3_5_pump_latency_highlow.xlsx', index=False)

Q3-6. PsychoPy: Did mean pumps change significantly more in the High experimental group than the Low experimental group from Pre to Post?

**Analysis:** Independent t-test on mean_pumps_change (High vs Low)


In [ ]:
ttest(bart_comparison_exp[bart_comparison_exp['usage_group']=='High']['mean_pumps_change'],
      bart_comparison_exp[bart_comparison_exp['usage_group']=='Low']['mean_pumps_change'], ('High','Low'))
bart_comparison_exp[['participant_num','group','usage_group','mean_pumps_exp1','mean_pumps_exp2','mean_pumps_change']].to_excel(RESULTS_DIR / 'q3_6_mean_pumps_highlow.xlsx', index=False)

Q3-7. PsychoPy: Did explosion rate change significantly more in the High experimental group than the Low experimental group from Pre to Post?

**Analysis:** Independent t-test on explosion_rate_change (High vs Low)


In [ ]:
ttest(bart_comparison_exp[bart_comparison_exp['usage_group']=='High']['explosion_rate_change'],
      bart_comparison_exp[bart_comparison_exp['usage_group']=='Low']['explosion_rate_change'], ('High','Low'))
bart_comparison_exp[['participant_num','group','usage_group','explosion_rate_exp1','explosion_rate_exp2','explosion_rate_change']].to_excel(RESULTS_DIR / 'q3_7_explosion_rate_highlow.xlsx', index=False)

App data analysis: estimating shortform viewing time-of-day from game session time-of-day
Assumption: game session time-of-day = shortform (Reels/Shorts) viewing time-of-day.
Compared against self-reported usual bedtime to check shortform viewing patterns before sleep.


In [ ]:
# ── Session grouping (a 25-minute gap starts a new session) ──────────────────
game_raw = pd.read_excel(DATA_DIR / 'GameResults_cleaned.xlsx')
game_raw['timestamp'] = pd.to_datetime(game_raw['timestamp'])

# Exclude exp sessions, extract participant_id, experimental group only
game_base2 = game_raw[~game_raw['participantName'].str.contains('exp', na=False)].copy()
game_base2['participant_id'] = game_base2['participantName'].str.extract(r'^(\d+)_')
game_base2['participant_num'] = pd.to_numeric(game_base2['participant_id'], errors='coerce')
game_base2 = game_base2[game_base2['participant_num'].isin(EXPERIMENTAL_GROUP)]

# A gap of 25+ minutes -> new session
game_base2 = game_base2.sort_values(['participant_id', 'timestamp'])
game_base2['gap'] = game_base2.groupby('participant_id')['timestamp'].diff().dt.total_seconds()
game_base2['new_session'] = (game_base2['gap'].isna()) | (game_base2['gap'] > 25 * 60)
game_base2['session_id'] = game_base2.groupby('participant_id')['new_session'].cumsum()

sessions = (
    game_base2.groupby(['participant_id', 'session_id'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'session_time'})
)
sessions['hour'] = sessions['session_time'].dt.hour + sessions['session_time'].dt.minute / 60

print(f"Total records: {len(game_base2)} rows -> sessions: {len(sessions)}, participants: {sessions['participant_id'].nunique()}")

# ── Bedtime/wake-up time mapping ───────────────────────────────────────────
preform2 = pd.read_excel(DATA_DIR / 'googleform' / 'BRAIN-PT_preform.xlsx')
name_col2 = '성함을 입력해주세요. (ex. 홍길동) '
preform2['participant_id'] = preform2[name_col2].str.extract(r'^(\d+)_')
preform2['participant_num'] = pd.to_numeric(preform2['participant_id'], errors='coerce')
preform2 = preform2[preform2['participant_num'].isin(EXPERIMENTAL_GROUP)]

sleep_col2  = '평소 취침 시간을 선택해주세요.'
wakeup_col2 = '평소 기상 시간을 선택해주세요.'

bedtime_map2 = {
    '23시 이전':                        22.5,
    '23시 이후 24시 이전':               23.5,
    '24시 이후 1시(새벽) 이전':           0.5,
    '1시(새벽) 이전 2시(새벽) 이후':      1.5,
    '2시(새벽) 이전 3시(새벽) 이후':      2.5,
    '3시(새벽) 이후':                     3.5,
}
wakeup_map2 = {
    '6시 이전':        5.5,
    '6시 이후 8시 이전':  7.0,
    '8시 이후 10시 이전': 9.0,
    '10시 이후 12시 이전':11.0,
}

preform2['bedtime_clock']  = preform2[sleep_col2].map(bedtime_map2)
preform2['wakeup_clock']   = preform2[wakeup_col2].map(wakeup_map2)

bedtime_info = preform2[['participant_id', sleep_col2, wakeup_col2, 'bedtime_clock', 'wakeup_clock']].dropna(subset=['participant_id'])
sessions2 = sessions.merge(bedtime_info, on='participant_id', how='left')

print("\nBedtime/wake-up time mapping:")
print(bedtime_info[['participant_id','bedtime_clock','wakeup_clock']].sort_values('bedtime_clock').to_string(index=False))

In [ ]:
# ── Bedtime-sorted spaghetti: play sessions + bedtime/wakeup markers ──
order = (
    bedtime_info.sort_values('bedtime_clock')['participant_id']
    .dropna().unique().tolist()
)
rest = [p for p in sorted(sessions2['participant_id'].dropna().unique()) if p not in order]
order = order + rest

fig, ax = plt.subplots(figsize=(14, max(6, len(order) * 0.6)))

for y_idx, pid in enumerate(order):
    sub = sessions2[sessions2['participant_id'] == pid]
    if sub.empty:
        continue

    ax.scatter(sub['hour'], [y_idx] * len(sub),
               s=15, alpha=0.45, color='steelblue', zorder=3)

    row = bedtime_info[bedtime_info['participant_id'] == pid]
    if row.empty:
        continue

    bt = row['bedtime_clock'].values[0]
    if pd.notna(bt):
        ax.plot([bt, bt], [y_idx - 0.4, y_idx + 0.4],
                color='crimson', linewidth=2.5, zorder=4, label='Bedtime' if y_idx == 0 else '')

    wt = row['wakeup_clock'].values[0]
    if pd.notna(wt):
        ax.plot([wt, wt], [y_idx - 0.4, y_idx + 0.4],
                color='darkorange', linewidth=2.5, zorder=4, label='Wake-up time' if y_idx == 0 else '')

ax.set_yticks(range(len(order)))
ax.set_yticklabels(
    [f"P{pid}  (bed {bedtime_info[bedtime_info['participant_id']==pid]['bedtime_clock'].values[0]:.1f}h / wake {bedtime_info[bedtime_info['participant_id']==pid]['wakeup_clock'].values[0]:.1f}h)"
     if pid in bedtime_info['participant_id'].values else f"P{pid}"
     for pid in order],
    fontsize=8
)
ax.set_xticks(range(0, 25, 1))
ax.set_xticklabels([f'{h:02d}:00' if h % 3 == 0 else '' for h in range(0, 25)], fontsize=8)
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Participant (sorted by bedtime)', fontsize=11)
ax.set_title(
    'Game Session Time Distribution — Experimental Group\n'
    'Blue dots = actual play sessions  |  Red = reported bedtime  |  Orange = reported wake-up time',
    fontsize=12
)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Figures 

In [ ]:
COLORS = {'Control': '#2a78d6', 'Experimental': '#eb6834'}

def plot_pre_post(ax, data, pre_col, post_col, ylabel, title):
    """Pre->Post line plot: group mean +/- SEM (thick line) + individual trajectories (thin background lines), with paired t-test significance"""
    sig_parts = []
    for group in ['Control', 'Experimental']:
        color = COLORS[group]
        sub = data[data['group'] == group]
        for _, row in sub.iterrows():
            ax.plot([0, 1], [row[pre_col], row[post_col]], color=color, alpha=0.15, linewidth=1, zorder=1)
        means = [sub[pre_col].mean(), sub[post_col].mean()]
        sems = [sub[pre_col].sem(), sub[post_col].sem()]
        ax.errorbar([0, 1], means, yerr=sems, color=color, linewidth=2.5,
                    marker='o', markersize=9, capsize=4, label=f'{group} (n={len(sub)})', zorder=3)
        t, p = stats.ttest_rel(sub[pre_col], sub[post_col])
        star = '**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.')
        sig_parts.append(f'{group}: p={p:.3f} {star}')

    ax.set_xlim(-0.3, 1.3)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Pre', 'Post'], fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=30)
    ax.text(0.5, 1.06, '    '.join(sig_parts), ha='center', va='bottom', fontsize=9,
            color='#52514e', transform=ax.transAxes)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', labelsize=10)
    ax.legend(frameon=False, fontsize=9, loc='lower left' if means[0] > means[1] else 'upper left')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_pre_post(ax, matched_data, 'total_shortform_minutes_pre', 'total_shortform_minutes_post',
              'Shortform usage (min/day)', 'Shortform Usage Time: Pre vs Post')
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(RESULTS_DIR / 'fig_shortform_pre_post.png', dpi=200)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_pre_post(ax, adhd_matched, 'adhd_score_pre', 'adhd_score_post',
              'ADHD score', 'ADHD Score: Pre vs Post')
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(RESULTS_DIR / 'fig_adhd_pre_post.png', dpi=200)
plt.show()
